# Notebook for running the winning model




In [1]:
from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

In [2]:
def summarise(label, subset):
    n_true_pos = subset["y_true"].sum()
    n_caught = subset[(subset["y_true"] == 1) & (subset["y_pred"] == 1)].shape[0]
    recall = n_caught / n_true_pos if n_true_pos else float("nan")
    n_pred_pos = subset["y_pred"].sum()
    precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
    print(
        f"{label}: {len(subset)} rows, {n_true_pos} true escalations, "
        f"{n_caught} caught -> recall={recall:.3f}, precision={precision:.3f}"
    )

In [3]:
def run_model(config, params):
    data_sources = [
        src
        for src, include in zip(
            ["food", "rain", "text"],
            [
                config["include_food"],
                config["include_rain"],
                config["include_text"],
            ],
        )
        if include
    ]

    model_data, predictor_cols = get_clean_combined_data(
        data_sources=data_sources,
        k=config["k"],
        event_col=config["event_col"],
        conflict_only_embeddings=config["conflict_only"],
    )

    final_params = {
        **params,
        "k": config["k"],
        "event_col": config["event_col"],
        "n_splits": config["n_splits"],
        "use_pca": config["use_pca"],
    }

    results, best_params, shap_importance, onset_predictions = train_evaluate_model(
        model_data,
        predictor_cols,
        final_params,
        best_params=True,  # skip RandomizedSearchCV
        use_pca=config["use_pca"],
        compute_shap=True,
        shap_sample_size=2000,
        return_onset_predictions=True,
    )
    return results, best_params, shap_importance, onset_predictions

In [4]:
def model_report(config, params):
    results, best_params, shap_importance, onset_predictions = run_model(config, params)
    print("========MODEL REPORT========")
    print("---Results\n")
    print(results)
    print("---Best Params\n")
    print(best_params)
    print("---SHAP Importance\n")
    print(shap_importance)
    
    onset_predictions["year_month"] = onset_predictions["year_month"].astype(str)
    war_outbreak = "2023-04"
    print("Pre and post war:")
    pre_war = onset_predictions[onset_predictions["year_month"] < war_outbreak]
    post_war = onset_predictions[onset_predictions["year_month"] >= war_outbreak]

    summarise("Pre-war  (Jan-Mar 2023)", pre_war)
    summarise("Post-war (Apr-Dec 2023)", post_war)
    
    key_regions = ["Khartoum", "North Darfur", "South Darfur", "West Darfur",
                   "Central Darfur", "East Darfur", "West Kordofan", "South Kordofan"]

    print("-----Key war-affected regions\n")
    key_region_rows = onset_predictions[onset_predictions["region"].isin(key_regions)]
    summarise("Key regions (all onset months)", key_region_rows)

    for region in key_regions:
        region_rows = onset_predictions[onset_predictions["region"] == region]
        if region_rows["y_true"].sum() > 0:
            summarise(f"  {region}", region_rows)
    

## Pipeline A

In [5]:
# --- Model A: numeric-only baseline (ACLED + food + rain, no text) ---
model_a_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": False,
    "conflict_only": None,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 4,
    "use_pca": False,
}

# (acled_sub_food_rain_threshold_change_1.75_4, onset_aupr=0.3379, active_aupr=0.2242)
model_a_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 3,
    "max_delta_step": 5,
    "gamma": 5,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 10,
    "colsample_bylevel": 0.8,
}

In [6]:
model_report(model_a_config, model_a_xgb_params)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: C:\Users\evely\OneDrive\Documents\GitHub\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: C:\Users\evely\OneDrive\Documents\GitHub\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: C:\Users\evely\.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:HDXClient:Reading local file: data/hdx\hdx_sudan_food_prices.csv
INFO:Name Mapping:Renamed

--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------
========MODEL REPORT========
---Results

{'optimal_threshold': '0.4578', 'n_predictors': 31, 'onset_aupr': '0.3254', 'onset_precision_class1': '0.3059', 'onset_recall_class1': '0.5306', 'onset_f1_class1': '0.3881', 'active_aupr': '0.2668', 'active_precision_class1': '0.2375', 'active_recall_class1': '0.5429', 'active_f1_class1': '0.3304'}
---Best Params

{'max_depth': 3, 'min_child_weight': 3, 'max_delta_step': 5, 'gamma': 5, 'learning_rate': 0.01, 'subsample'

## Model B

In [7]:
# --- Model B: numeric + food + rain + text (matched to Model A) ---
model_b_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": True,
    "conflict_only": False, #TODO change this when I have a good computer
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": True,
}

# (acled_sub_food_rain_text_conflict_pca_1.75_5, onset_aupr=0.3941, active_aupr=0.2155)
model_b_xgb_params = {
    "max_depth": 7,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 3,
    "learning_rate": 0.01,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 2.0,
    "reg_lambda": 10,
    "colsample_bylevel": 0.8,
}

In [8]:
model_report(
    model_b_config, model_b_xgb_params
)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:HDXClient:Reading local file: data/hdx\hdx_sudan_food_prices.csv
INFO:Name Mapping:Renamed region 'Eastern Darfur' to 'East Darfur' (manual override)
INFO:Name Mapping:Renamed region 'Al Gezira' to 'Al Jazirah' (manual override)
INFO:Name Mapping:Renamed region 'Nile' to 'River Nile' (manual override)
INFO:Data preparation:Food prices data processed.
INFO:Name Mapping:Creating a PCODE map for rainfall data. This may take a few seconds...
INFO:HDXClient:Reading local file data/hdx/admin_boundaries\hdx_sdn_admin1.geojson
INFO:HDXClient:Reading local file: data/hdx\hdx_sdn_rainfall_subnat_full.csv
INFO:Name Mapping:Renamed region 'Aj Jazirah' to 'Al Jazirah' (manual override)
INFO:Rainfal

--- Fold 1 ---
Train window: 2018-01 to 2018-10 (180 rows)
Test window:  2018-11 to 2019-08 (180 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (360 rows)
Test window:  2019-09 to 2020-06 (180 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (540 rows)
Test window:  2020-07 to 2021-04 (180 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (720 rows)
Test window:  2021-05 to 2022-02 (180 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (900 rows)
Test window:  2022-03 to 2022-12 (180 rows)
------------------------------
========MODEL REPORT========
---Results

{'optimal_threshold': '0.3460', 'n_predictors': 75, 'onset_aupr': '0.3646', 'onset_precision_class1': '0.2468', 'onset_recall_class1': '0.7959', 'onset_f1_class1': '0.3768', 'active_aupr': '0.2840', 'active_precision_class1': '0.1799', 'active_recall_class1': '0.9714', 'active_f1_class1':

In [9]:
from utils.constants import ACTIVE_START_DATE, ACTIVE_END_DATE
import pandas as pd

active_prevalence_records = []
for k_test in [1.75]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[], k=k_test, event_col="sub_event_type", conflict_only_embeddings=True,
    )
    active_slice = model_data[
        (model_data["year_month"] >= pd.Period(ACTIVE_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ACTIVE_END_DATE, freq="M"))
    ]
    n_total = len(active_slice)
    n_escalations = int(active_slice["target_escalation"].sum())
    active_prevalence_records.append({
        "k": k_test,
        "active_prevalence_percentage": round(n_escalations / n_total * 100, 1),
        "n_escalations": n_escalations,
        "n_active_rows": n_total,
    })

active_prevalence_df = pd.DataFrame(active_prevalence_records).set_index("k")
print(active_prevalence_df)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.


      active_prevalence_percentage  n_escalations  n_active_rows
k                                                               
1.75                          16.2             35            216


### AUPR relative to baseline

Raw AUPR values at k=1.75 are lower than at k=1.0 (Model A onset: 0.3379 vs 0.3213 previously;
active: 0.2242 vs 0.3693 previously), which could look like a regression. However, AUPR's
no-skill baseline equals the positive-class prevalence, which is itself lower at the stricter
k=1.75 threshold (22.7% onset, 16.2% active, vs 30.6% onset at k=1.0). Expressed as a lift over
that baseline, Model A achieves 1.49x on onset and 1.38x on active, a genuine, meaningful
improvement over chance, and the onset lift is comparable to or better than what was observed at
k=1.0. The apparent decline in raw AUPR is therefore largely explained by evaluating against a
stricter, more legitimate target, not by weaker model performance. [Model B's equivalent lift
figures are not yet available and require re-running with local embeddings access.]


### Regional performance (Model A, numeric-only, k=1.75)

Onset-window recall varies substantially by region. Notably, the model caught only 1 of 5 true
escalations in Khartoum (recall 0.200) - the capital and the site of the actual April 2023
outbreak - making this the weakest result among the war's key regions and arguably the most
consequential, since it is the flashpoint the whole project is framed around. Performance was
stronger in North/East Darfur (recall 1.0, though based on only 1 and 3 true cases respectively)
and West Kordofan (4 of 4 caught). This pattern is tentative given very small per-region sample
sizes (1-5 true escalations per region), but is consistent with a plausible story: the model may
be more sensitive to escalation spreading to secondary regions than to detecting the originating
flashpoint itself. This should be re-checked once Model B's regional breakdown is available, to
see whether text specifically improves Khartoum detection - the case most directly relevant to
the project's central hypothesis.

